# Trial curvature geometry

Source-node and symmetry checks for the two Ti_Q_2A interpolants (trial03, trial04). The raw-MMN check (`inputs/check_raw_mmn_wilson.ipynb`) establishes equality of the discrete occupied subspaces and Wilson loops; this notebook asks two separate downstream questions:

- how different are the continuous curvatures evaluated at every source-mesh node; and
- does each interpolant separately obey the surviving crystal symmetries (T, C4z, Mx) away from those nodes?

The traced Cartesian curvature is the axial vector `(Omega_yz, Omega_zx, Omega_xy)`. For an orthogonal spatial operation `O` it transforms as `det(O) O Omega`; time reversal sends `Omega(k)` to `-Omega(-k)`.

Set `RUN_COMPUTATION = False` to load the last saved run from `results/trial_geometry/`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import STO_AA_CACHE
from validation.symmetry.compare_trial_curvature_lib import (
    PREFIX,
    TRIALS,
    build_trial,
    comparison_metrics,
    evaluate_trial,
)
from validation.symmetry.check_trial_curvature_geometry_lib import (
    OUTPUT,
    symmetry_checks,
)
from wannier.wannier_io import read_wannier_checkpoint

## Config

`TRIALS` (imported above) is the default trial03/trial04 directory/source/cache setup from `compare_trial_curvature_lib`; override it in place if you want to point at different runs (mirrors the old `--trial03-directory`/`--trial04-source`/etc. flags).

In [ ]:
RUN_COMPUTATION = False  # True: rebuild both trials and recompute. False: load results/trial_geometry/.
OUTPUT_DIR = OUTPUT  # results/trial_geometry
GENERIC_SEED = 20260827
N_GENERIC_POINTS = 24

# Example override (mirrors --trial03-directory/--trial03-source/etc.):
# TRIALS["trial03"] = {"directory": Path("..."), "mmn_directory": Path("..."), "source": "mmn",
#                      "label": "...", "color": "C0", "cache": STO_AA_CACHE / "..."}

## Build both trials, evaluate on the source mesh, and check off-grid symmetry

`source_results[trial]["curvature"]` is the full traced-occupied curvature tensor on the source checkpoint's k-mesh; `symmetries[trial]` holds the T/C4z/Mx covariance residuals.

In [ ]:
if RUN_COMPUTATION:
    checkpoint = read_wannier_checkpoint(
        Path(TRIALS["trial03"]["directory"]) / f"{PREFIX}.chk"
    )
    source_points = np.asarray(checkpoint["kpt_red"], dtype=float)
    mp_grid = tuple(int(value) for value in checkpoint["mp_grid"])

    rng = np.random.default_rng(GENERIC_SEED)
    generic_points = rng.uniform(0.071, 0.929, size=(N_GENERIC_POINTS, 3))

    source_results = {}
    symmetries = {}
    for trial in ("trial03", "trial04"):
        model, pos = build_trial(trial)
        source_results[trial] = evaluate_trial(model, pos, source_points)
        symmetries[trial] = symmetry_checks(model, pos, generic_points)

## Compare the two trials on the source mesh

In [ ]:
if RUN_COMPUTATION:
    comparison = comparison_metrics(source_results)
    comparison["point_set"] = (
        f"all {len(source_points)} source checkpoint nodes on "
        f"{mp_grid[0]}x{mp_grid[1]}x{mp_grid[2]}"
    )
    summary = {
        "source_mesh_trial_comparison": comparison,
        "offgrid_symmetry": symmetries,
        "symmetry_conventions": {
            "curvature_vector": "(Tr Omega_yz, Tr Omega_zx, Tr Omega_xy)",
            "spatial_rule": "omega(O k) = det(O) O omega(k)",
            "time_reversal_rule": "omega(-k) = -omega(k)",
            "generic_points": len(generic_points),
        },
    }
    print(json.dumps(summary, indent=2, sort_keys=True))

## Save

In [ ]:
if RUN_COMPUTATION:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = OUTPUT_DIR / "trial_curvature_geometry_summary.json"
    data_path = OUTPUT_DIR / "trial_curvature_geometry.npz"
    summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    np.savez_compressed(
        data_path,
        source_kpoints_reduced=source_points,
        generic_kpoints_reduced=generic_points,
        trial03_source_trace=source_results["trial03"]["trace"],
        trial04_source_trace=source_results["trial04"]["trace"],
        trial03_source_energies=source_results["trial03"]["energies"],
        trial04_source_energies=source_results["trial04"]["energies"],
    )
    print(f"saved {summary_path}")
    print(f"saved {data_path}")

## Load previously saved results (skip if you just computed above)

Only the trace/energy arrays round-trip through the `.npz` (not the full curvature tensor or the off-grid symmetry points); the summary JSON has every metric.

In [ ]:
if not RUN_COMPUTATION:
    summary = json.loads((OUTPUT_DIR / "trial_curvature_geometry_summary.json").read_text())
    data = np.load(OUTPUT_DIR / "trial_curvature_geometry.npz")
    source_results = {
        "trial03": {"trace": data["trial03_source_trace"], "energies": data["trial03_source_energies"]},
        "trial04": {"trace": data["trial04_source_trace"], "energies": data["trial04_source_energies"]},
    }
    source_points = data["source_kpoints_reduced"]
    generic_points = data["generic_kpoints_reduced"]
    print(json.dumps(summary, indent=2, sort_keys=True))